# Boosting: AdaBoost Classification

This notebook demonstrates AdaBoost (Adaptive Boosting), which sequentially fits weak learners and focuses on misclassified instances.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
n_estimators_list = [1, 5, 10, 50, 100, 200]
train_scores = []
test_scores = []

for n in n_estimators_list:
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), 
                             n_estimators=n, random_state=42, learning_rate=1.0)
    ada.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, ada.predict(X_train))
    test_acc = accuracy_score(y_test, ada.predict(X_test))
    
    train_scores.append(train_acc)
    test_scores.append(test_acc)
    
    print(f"n_estimators={n:3d}: Train={train_acc:.4f}, Test={test_acc:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(n_estimators_list, train_scores, 'o-', label='Training', linewidth=2, markersize=8)
plt.plot(n_estimators_list, test_scores, 's-', label='Testing', linewidth=2, markersize=8)
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('AdaBoost: Training Progress')
plt.legend()
plt.grid(alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Train final AdaBoost
ada_final = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                               n_estimators=100, random_state=42, learning_rate=1.0)
ada_final.fit(X_train, y_train)

y_pred = ada_final.predict(X_test)

print("\n=== ADABOOST CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - AdaBoost')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
importances = ada_final.feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), indices)
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances - AdaBoost')
plt.tight_layout()
plt.show()

print(f"\nTop 5 important features indices: {np.argsort(importances)[-5:]}")

In [ ]:
print("""
AdaBoost Classification Summary:
- Sequential boosting: fits weak learners iteratively
- Focuses on misclassified samples (increases weights)
- Typically uses decision stumps (max_depth=1)
- Learning rate controls contribution of each estimator
- Final prediction: weighted majority vote
- Good with imbalanced data (focuses on hard examples)
- Can suffer from noise (sensitive to outliers)
""")